In [ ]:
import requests
import os
from pathlib import Path

data_dir = Path("data/raw")
data_dir.mkdir(parents=True, exist_ok=True)

taxi_file = data_dir / "taxi_data.parquet"
zone_file = data_dir / "taxi_zone_lookup.csv"

# Only download taxi data if not already present
if not taxi_file.exists():
    url1 = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-01.parquet"
    response1 = requests.get(url1)
    response1.raise_for_status()
    taxi_file.write_bytes(response1.content)
    print("Taxi data downloaded.")
else:
    print("Taxi data already exists. Skipping download.")

# Only download zone lookup if not already present
if not zone_file.exists():
    url2 = "https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv"
    response2 = requests.get(url2)
    response2.raise_for_status()
    zone_file.write_bytes(response2.content)
    print("Zone lookup downloaded.")
else:
    print("Zone lookup already exists. Skipping download.")

print("Download check complete!")

## Part 1: Distributed Data Processing with Spark

Task 1.1

In [ ]:
import os
import sys
import subprocess

os.environ["JAVA_HOME"] = r"C:\Program Files\Java\jdk-17"
os.environ["HADOOP_HOME"] = r"C:\hadoop"
os.environ["hadoop.home.dir"] = r"C:\hadoop"
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable
os.environ["PATH"] = (
    r"C:\Program Files\Java\jdk-17\bin;"
    r"C:\hadoop\bin;"
    + os.environ["PATH"]
)

print("JAVA_HOME:", os.environ["JAVA_HOME"])
print("HADOOP_HOME:", os.environ["HADOOP_HOME"])
print("PYSPARK_PYTHON:", os.environ["PYSPARK_PYTHON"])
print("Python exe:", sys.executable)
print("winutils exists:", os.path.exists(r"C:\hadoop\bin\winutils.exe"))
print(subprocess.check_output(["java", "-version"], stderr=subprocess.STDOUT).decode())

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .master("local[1]") \
    .appName("TaxiAnalysis") \
    .config("spark.executor.memory", "2g") \
    .config("spark.driver.memory", "2g") \
    .config("spark.sql.adaptive.enabled", "true") \
    .config("spark.python.worker.reuse", "false") \
    .config("spark.sql.shuffle.partitions", "1") \
    .getOrCreate()

df = spark.read.parquet("data/raw/taxi_data.parquet")
df.printSchema()

In [ ]:
row_count = df.count()
partition_count = df.rdd.getNumPartitions()

print("Rows:", row_count)
print("Partitions:", partition_count)

In [ ]:
import time
import pandas as pd

# Pandas load time
start = time.time()
df = pd.read_parquet("data/raw/taxi_data.parquet")
pandas_time = time.time() - start

# Spark load time
start = time.time()
df = spark.read.parquet("data/raw/taxi_data.parquet")
df.count()
spark_time = time.time() - start

print(f"Spark load time: {spark_time:.2f} seconds")
print(f"Pandas load time: {pandas_time:.2f} seconds")

task 1.2

In [ ]:
from pyspark.sql import functions as F

initial_count = df.count()
print(f"Initial rows: {initial_count}")

# setting critical columns
critical_cols = [
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "PULocationID",
    "DOLocationID",
    "fare_amount",
    "trip_distance",
    "tip_amount"
]

# drop nulls
df_clean = df.dropna(subset=critical_cols)
after_nulls = df_clean.count()
print(f"Rows after null removal: {after_nulls}")
print(f"Rows removed for nulls: {initial_count - after_nulls}")


# clean rows
df_clean = df_clean.filter(F.col("trip_distance") > 0)
after_distance = df_clean.count()
print(f"Rows removed for non-positive distance: {after_nulls - after_distance}")

df_clean = df_clean.filter(F.col("fare_amount") >= 0)
after_negative_fare = df_clean.count()
print(f"Rows removed for negative fare: {after_distance - after_negative_fare}")

df_clean = df_clean.filter(F.col("fare_amount") <= 500)
after_high_fare = df_clean.count()
print(f"Rows removed for fare > 500: {after_negative_fare - after_high_fare}")

df_clean = df_clean.filter(
    F.col("tpep_dropoff_datetime") >= F.col("tpep_pickup_datetime"))
after_bad_time = df_clean.count()
print(f"Rows removed for dropoff before pickup: {after_high_fare - after_bad_time}")

Derived Columns

In [ ]:
from pyspark.sql import functions as F

df_clean = df_clean.withColumn(
    "trip_duration_minutes",
    (
        F.unix_timestamp("tpep_dropoff_datetime") -
        F.unix_timestamp("tpep_pickup_datetime")
    ) / 60.0)

df_clean = df_clean.withColumn(
    "trip_speed_mph",
    F.when(
        F.col("trip_duration_minutes") > 0,
        F.col("trip_distance") / (F.col("trip_duration_minutes") / 60.0)
    ).otherwise(0.0))

df_clean = df_clean.withColumn(
    "pickup_hour",
    F.hour("tpep_pickup_datetime"))

df_clean = df_clean.withColumn(
    "pickup_day_of_week",
    F.dayofweek("tpep_pickup_datetime"))

df_clean = df_clean.withColumn(
    "tip_percentage",
    F.when(
        F.col("fare_amount") > 1, 
        (F.col("tip_amount") / F.col("fare_amount")) * 100.0
    ).otherwise(0.0))

final_count = df_clean.count()
print(f"Final cleaned rows: {final_count}")

df_clean.select(
    "trip_duration_minutes",
    "trip_speed_mph",
    "pickup_hour",
    "pickup_day_of_week",
    "tip_percentage"
).show(5, truncate=False)

In [ ]:
df_clean.select(
    F.min("trip_duration_minutes"),
    F.min("trip_speed_mph"),
    F.min("tip_percentage"),
    F.max("tip_percentage")
).show()

task 1.3

In [ ]:
df_clean.createOrReplaceTempView("taxi_trips")

Query 1: What are the top 10 busiest pickup hours, and what is the average fare and tip 
percentage for each? 

In [ ]:
query1 = """
SELECT
    pickup_hour,
    COUNT(*) AS trip_count,
    ROUND(AVG(fare_amount), 2) AS avg_fare,
    ROUND(AVG(tip_percentage), 2) AS avg_tip_percentage
FROM taxi_trips
GROUP BY pickup_hour
ORDER BY trip_count DESC
LIMIT 10
"""

busiest_hours = spark.sql(query1)
busiest_hours.show(truncate=False)

Busiest hours are in the afternoon into the evening (from 15:00 to 19:00), being usual rush hour travel patterns. While the average fare is relatively consistent, the tip percentages seem to trend higher as it gets later into the pickup hours.

Query 2: Which day of the week has the highest average trip speed? Include average distance and duration. 

In [ ]:
query2 = """
SELECT
    pickup_day_of_week,
    ROUND(AVG(trip_speed_mph), 2) AS avg_speed_mph,
    ROUND(AVG(trip_distance), 2) AS avg_distance,
    ROUND(AVG(trip_duration_minutes), 2) AS avg_duration_minutes
FROM taxi_trips
GROUP BY pickup_day_of_week
ORDER BY avg_speed_mph DESC
"""

highest_avg_trip_speed = spark.sql(query2)
highest_avg_trip_speed.show(truncate=False)

Tuesdays seem to have a good flow of traffic leading to efficient travel, when compared to the similar trip duration of days like Wenesday and Thursday, that have a much lower avg speed and fairly shorter average distance but ultimately a very similar average trip duration.

Query 3: Using a window function, rank the top 5 pickup locations by total revenue for each day of the week.

In [ ]:
query3 = """
WITH revenue_by_day AS (
    SELECT
        pickup_day_of_week,
        PULocationID,
        ROUND(SUM(total_amount), 2) AS total_revenue
    FROM taxi_trips
    GROUP BY pickup_day_of_week, PULocationID
),
ranked_locations AS (
    SELECT
        pickup_day_of_week,
        PULocationID,
        total_revenue,
        ROW_NUMBER() OVER (
            PARTITION BY pickup_day_of_week
            ORDER BY total_revenue DESC
        ) AS revenue_rank
    FROM revenue_by_day
)
SELECT
    pickup_day_of_week,
    PULocationID,
    total_revenue,
    revenue_rank
FROM ranked_locations
WHERE revenue_rank <= 5
ORDER BY pickup_day_of_week, revenue_rank
"""

top_5_PULocations = spark.sql(query3)
top_5_PULocations.show(50, truncate=False)

Pickup locations IDed 132 followed by 138 for each day of the week have the highest revenues. Location 132 by a large margin at that, with cases such as on Saturday, you can see that 132 has about 3 times the revenue as the next pickup Location below it.

Query 4: Calculate the cumulative trip count by hour of day (running total from hour 0 to 23). At what hour does the cumulative count surpass 50% of daily trips? 

In [ ]:
query4 = """
WITH hourly_counts AS (
    SELECT
        pickup_hour,
        COUNT(*) AS trip_count
    FROM taxi_trips
    GROUP BY pickup_hour
),
cumulative_counts AS (
    SELECT
        pickup_hour,
        trip_count,
        SUM(trip_count) OVER (
            ORDER BY pickup_hour
            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        ) AS cumulative_trip_count,
        SUM(trip_count) OVER () AS total_trip_count
    FROM hourly_counts
)
SELECT
    pickup_hour,
    trip_count,
    cumulative_trip_count,
    ROUND(cumulative_trip_count * 100.0 / total_trip_count, 2) AS cumulative_pct
FROM cumulative_counts
ORDER BY pickup_hour
"""

cumulative_trip_count = spark.sql(query4)
cumulative_trip_count.show(24, truncate=False)

In [ ]:
cumulative_trip_count.filter("cumulative_pct > 50").orderBy("pickup_hour").show(1, truncate=False)

It takes until about the start of afternoon rush hour to accumulate half of the day's trips. Which means that more than half the day has to pass before it gets to see half of it's demand for transportation.

Query 5: Compare average fare, distance, and tip percentage between short trips (<2 miles), medium trips (2–10 miles), and long trips (>10 miles). Which category has the highest tip percentage? 

In [ ]:
query5 = """
SELECT
    CASE
        WHEN trip_distance < 2 THEN 'Short (<2 miles)'
        WHEN trip_distance <= 10 THEN 'Medium (2-10 miles)'
        ELSE 'Long (>10 miles)'
    END AS trip_category,
    COUNT(*) AS trip_count,
    ROUND(AVG(fare_amount), 2) AS avg_fare,
    ROUND(AVG(trip_distance), 2) AS avg_distance,
    ROUND(AVG(tip_percentage), 2) AS avg_tip_percentage
FROM taxi_trips
GROUP BY
    CASE
        WHEN trip_distance < 2 THEN 'Short (<2 miles)'
        WHEN trip_distance <= 10 THEN 'Medium (2-10 miles)'
        ELSE 'Long (>10 miles)'
    END
ORDER BY avg_distance
"""

avg_distance_fare_tip = spark.sql(query5)
avg_distance_fare_tip.show(truncate=False)

The average distance for medium trips trends closer to the beginning of the threshold, showing that most people take shorter trips when combined with the amount of trips for short, but the average distance for long clears the beginning of the threshold number by a fairly large amount. This fact is not helped by the fact that this threshold, would be lumped with any large distance outliers.

Task 1.4

In [ ]:
import time

start = time.time()
spark.sql(query1).show()
no_cache_time = time.time() - start

print(f"No cache time: {no_cache_time:.2f} seconds")

In [ ]:
df_clean.cache()
df_clean.count()  

start = time.time()
spark.sql(query1).show()
cache_time = time.time() - start

print(f"With cache time: {cache_time:.2f} seconds")

In [ ]:
df_clean.write.mode("overwrite").partitionBy("pickup_hour").parquet("data/processed/taxi_partitioned")

df_partition = spark.read.parquet("data/processed/taxi_partitioned")

df_partition.filter("pickup_hour = 17").show(5)

spark.sql(query1).explain(True)